# Notebook 03 — Random Forest, Permutation Importance & OLS Comparison
**Part A — Temporal ML analysis**

## What this notebook does
1. Trains Random Forest (RF) and OLS regression on met → bioaerosol targets  
2. Evaluates both with **Leave-One-Out Cross-Validation (LOO-CV)**  
3. Reports R², RMSE, MAE for each model × target combination  
4. Computes **MDI feature importance** for RF  
5. Computes **Permutation Importance** (corrects MDI bias)  
6. Produces:
   - Fig 3 — MDI vs Permutation importance comparison (4 targets, 2×4 grid)
   - Fig 4 — Predictor ranking table (with agreement indicators)
   - Fig 5 — Observed vs predicted (OLS top row, RF bottom row)  
   - Table 8 — Model comparison (OLS vs RF)

## Key context
All 6 stations share **one weather station** → met variables explain temporal  
variation only, not spatial. Negative LOO-CV R² is structurally expected.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP — run this cell first if using Google Colab
# ═══════════════════════════════════════════════════════════════
import os, sys

# Option A: Clone the GitHub repo directly in Colab (recommended)
# !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git
# os.chdir('geoai-bioaerosol-prediction')

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

In [ ]:
# ── Install (uncomment on Colab) ──────────────────────────────────
# !pip install scikit-learn matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/df_analysis.csv')
for col in ['Aspergillus_conc', 'Alternaria_conc']:
    df[col] = df[col].fillna(df[col].median())

MET_FEATURES = ['GHI', 'Tamb', 'RH', 'WS', 'BP', 'sin_WD', 'cos_WD']
FEAT_LABELS  = ['GHI', 'Temp', 'RH', 'Wind Speed', 'Pressure', 'sin(WD)', 'cos(WD)']
TARGETS = {
    'pollen_conc':      ('Pollen',       '#378ADD'),
    'fungus_conc':      ('Total Fungus', '#D85A30'),
    'Aspergillus_conc': ('Aspergillus',  '#1D9E75'),
    'Alternaria_conc':  ('Alternaria',   '#BA7517'),
}

X  = df[MET_FEATURES].values
Xs = StandardScaler().fit_transform(X)   # standardised for OLS
print('Data loaded. n =', len(df))

## Step 1 — LOO-CV for both OLS and RF
### Why Leave-One-Out Cross-Validation?
With only n=64 observations, k-fold CV wastes too much data per fold.  
LOO-CV uses n-1 observations for training and 1 for testing at each step,  
maximising training data while still producing unbiased out-of-sample estimates.

### Why both OLS and RF?
Reviewers ask: "Why Random Forest?" — comparing against a simpler baseline  
answers this directly. If OLS performs similarly, RF is justified only by  
its interpretability (permutation importance, SHAP), not its predictive power.

In [ ]:
loo = LeaveOneOut()
results = {}

for tcol, (tlabel, col) in TARGETS.items():
    y    = df[tcol].values
    mask = ~np.isnan(y)
    Xm, Xsm, ym = X[mask], Xs[mask], y[mask]

    # ── OLS LOO-CV ──────────────────────────────────────────────
    y_ols = np.zeros_like(ym)
    for tr, te in loo.split(Xsm):
        lr = LinearRegression()
        lr.fit(Xsm[tr], ym[tr])
        y_ols[te] = lr.predict(Xsm[te])

    # ── RF LOO-CV ───────────────────────────────────────────────
    y_rf = np.zeros_like(ym)
    for tr, te in loo.split(Xm):
        rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        rf.fit(Xm[tr], ym[tr])
        y_rf[te] = rf.predict(Xm[te])

    results[tcol] = {
        'label': tlabel, 'color': col, 'y': ym,
        'y_ols': y_ols, 'y_rf': y_rf,
        'ols_r2':   r2_score(ym, y_ols),
        'ols_rmse': np.sqrt(mean_squared_error(ym, y_ols)),
        'ols_mae':  mean_absolute_error(ym, y_ols),
        'rf_r2':    r2_score(ym, y_rf),
        'rf_rmse':  np.sqrt(mean_squared_error(ym, y_rf)),
        'rf_mae':   mean_absolute_error(ym, y_rf),
    }
    print(f'{tlabel:20s}  OLS R²={results[tcol]["ols_r2"]:+.3f}  '
          f'RF R²={results[tcol]["rf_r2"]:+.3f}')

In [ ]:
# ── Table 8 — Model comparison ────────────────────────────────────
print('\nTable 8. OLS vs RF LOO-CV performance comparison')
print(f'{"Target":20s} {"OLS R²":>8} {"OLS RMSE":>10} {"OLS MAE":>9}  '
      f'{"RF R²":>8} {"RF RMSE":>10} {"RF MAE":>9}')
print('-'*90)
for tcol, res in results.items():
    print(f'{res["label"]:20s} {res["ols_r2"]:>8.3f} {res["ols_rmse"]:>10.1f} '
          f'{res["ols_mae"]:>9.1f}  {res["rf_r2"]:>8.3f} '
          f'{res["rf_rmse"]:>10.1f} {res["rf_mae"]:>9.1f}')
print()
print('Negative R² = met variables explain temporal variation only (not spatial).')
print('OLS slightly lower error on all targets → approximately linear relationships at this scale.')

## Step 2 — MDI and Permutation Importance
### What is MDI (Mean Decrease in Impurity)?
- Computed from the fully trained RF model  
- Measures how much each feature reduces node impurity when used as a split  
- **Known bias:** inflates importance for continuous high-cardinality features  
- Fast but can be misleading

### What is Permutation Importance (PI)?
- Randomly shuffles each feature's values 100 times  
- Measures how much model accuracy drops when a feature is scrambled  
- **Less biased** — directly measures a feature's contribution to accuracy  
- Slower but more reliable

### Interpretation rule used in the paper
Where MDI and PI agree → confident in that ranking  
Where they disagree → PI is taken as primary (e.g. Aspergillus: wind speed vs cos_WD)

In [ ]:
mdi_res  = {}
perm_res = {}

for tcol, (tlabel, col) in TARGETS.items():
    y    = df[tcol].values
    mask = ~np.isnan(y)
    Xm, ym = X[mask], y[mask]

    # Train final RF on all data (for importances)
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(Xm, ym)

    # MDI importance
    mdi_res[tcol] = rf.feature_importances_

    # Permutation importance — 100 repeats for stability
    pi = permutation_importance(rf, Xm, ym, n_repeats=100,
                                random_state=42, n_jobs=-1)
    perm_res[tcol] = {'mean': pi.importances_mean, 'std': pi.importances_std}

    # Print comparison
    print(f'\n{tlabel}:')
    print(f'  {"Feature":12s}  {"MDI rank":>10}  {"PI rank":>8}  {"Agreement":<10}')
    mdi_rank  = np.argsort(-mdi_res[tcol])
    perm_rank = np.argsort(-perm_res[tcol]['mean'])
    for feat_i, feat in enumerate(FEAT_LABELS):
        m_rank = list(mdi_rank).index(feat_i) + 1
        p_rank = list(perm_rank).index(feat_i) + 1
        agree  = '✓' if abs(m_rank - p_rank) <= 1 else '△ DIFFER'
        print(f'  {feat:12s}  {m_rank:>10}  {p_rank:>8}  {agree}')

In [ ]:
# ── Figure 3 — MDI vs Permutation Importance comparison ──────────
fig, axes = plt.subplots(4, 2, figsize=(13, 15))

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    mdi   = mdi_res[tcol]
    pmean = perm_res[tcol]['mean']
    pstd  = perm_res[tcol]['std']

    # Left column: MDI
    ax_m = axes[i, 0]
    ord_m = np.argsort(mdi)
    ax_m.barh([FEAT_LABELS[j] for j in ord_m], mdi[ord_m],
               color=col, alpha=0.82, edgecolor='white', height=0.6)
    for j, v in zip(ord_m, mdi[ord_m]):
        ax_m.text(v + 0.003, list(ord_m).index(j), f'{v:.3f}', va='center', fontsize=9)
    ax_m.set_xlim(0, mdi.max() * 1.25)
    ax_m.set_title(f'{tlabel} — MDI importance', fontsize=9, fontweight='bold')
    ax_m.set_xlabel('Mean decrease in impurity', fontsize=8)
    ax_m.spines[['top', 'right']].set_visible(False)

    # Right column: Permutation Importance with error bars
    ax_p = axes[i, 1]
    ord_p = np.argsort(pmean)
    ax_p.barh([FEAT_LABELS[j] for j in ord_p], pmean[ord_p],
               xerr=pstd[ord_p], color=col, alpha=0.75, edgecolor='white', height=0.6,
               error_kw=dict(elinewidth=0.9, capsize=3, color='#444'))
    for j, v in zip(ord_p, pmean[ord_p]):
        ax_p.text(max(v, 0) + 0.003, list(ord_p).index(j), f'{v:.3f}', va='center', fontsize=9)
    ax_p.set_xlim(0, max(pmean.max(), 0.01) * 1.35)
    ax_p.set_title(f'{tlabel} — Permutation importance (±SD, 100 repeats)', fontsize=9, fontweight='bold')
    ax_p.set_xlabel('Mean accuracy decrease', fontsize=8)
    ax_p.spines[['top', 'right']].set_visible(False)

fig.suptitle('Feature importance: MDI (left) vs Permutation importance (right)\n'
             'Permutation importance corrects MDI bias toward continuous high-cardinality features',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('fig_mdi_vs_permutation.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_mdi_vs_permutation.png')

In [ ]:
# ── Figure 4 — Predictor ranking table ───────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(17, 5))

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    ax = axes[i]
    ax.axis('off')

    mdi   = mdi_res[tcol]
    pmean = perm_res[tcol]['mean']

    # Rank by MDI
    rows = sorted(zip(FEAT_LABELS, mdi, pmean), key=lambda x: x[1], reverse=True)

    table_data = []
    for rank, (feat, m, p) in enumerate(rows, 1):
        perm_rank = sorted(range(len(pmean)),
                           key=lambda j: pmean[j], reverse=True).index(
                    FEAT_LABELS.index(feat)) + 1
        agree = '✓' if abs(rank - perm_rank) <= 1 else '△'
        table_data.append([str(rank), feat, f'{m:.3f}', f'{p:.3f} {agree}'])

    tbl = ax.table(cellText=table_data,
                   colLabels=['Rank', 'Feature', 'MDI', 'Perm ✓/△'],
                   loc='center', cellLoc='left')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.6)

    # Style header row
    for j in range(4):
        tbl[(0, j)].set_facecolor(col)
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    # Highlight top feature
    for j in range(4):
        tbl[(1, j)].set_facecolor('#F0F8FF')

    ax.set_title(f'{tlabel}', fontsize=10, fontweight='bold', pad=18)

fig.suptitle('Predictor ranking: MDI vs Permutation importance\n'
             '✓ = rankings agree within 1 position   '
             '△ = rankings differ (MDI bias likely)',
             fontsize=10, y=1.04)
plt.tight_layout()
plt.savefig('fig_ranking_table.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_ranking_table.png')

In [ ]:
# ── Figure 5 — Observed vs Predicted (OLS top row, RF bottom row) ─
fig, axes = plt.subplots(2, 4, figsize=(15, 9))
station_colors = plt.cm.tab10(np.linspace(0, 0.6, 6))

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    res  = results[tcol]
    stas = df['station'].values[~np.isnan(df[tcol].values)]

    for row, (model_label, yp) in enumerate([
        ('OLS', res['y_ols']),
        ('RF',  res['y_rf'])
    ]):
        ax = axes[row, i]
        y  = res['y']
        r2   = res[f'{model_label.lower()}_r2']
        rmse = res[f'{model_label.lower()}_rmse']
        mae  = res[f'{model_label.lower()}_mae']

        # Colour points by station
        for s in range(1, 7):
            idx = stas == s
            ax.scatter(y[idx], yp[idx],
                       color=station_colors[s-1], label=f'S{s}',
                       s=55, alpha=0.85, edgecolors='white',
                       linewidth=0.4, zorder=3)

        # 1:1 reference line
        mn, mx = min(y.min(), yp.min()), max(y.max(), yp.max())
        pad = (mx - mn) * 0.07
        ax.plot([mn-pad, mx+pad], [mn-pad, mx+pad], 'k--', lw=1, alpha=0.35)
        ax.set_xlim(mn-pad, mx+pad); ax.set_ylim(mn-pad, mx+pad)
        ax.set_xlabel('Observed (grain/m³)', fontsize=9)
        ax.set_ylabel('Predicted', fontsize=9)
        ax.set_title(f'{tlabel} — {model_label}', fontsize=10, fontweight='bold')

        # Metrics box
        ax.text(0.04, 0.96,
                f'R²   = {r2:.3f}\nRMSE = {rmse:.1f}\nMAE  = {mae:.1f}',
                transform=ax.transAxes, fontsize=8.5, va='top',
                bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                          edgecolor='#ccc', alpha=0.9))

        # Legend on first panel only
        if i == 0 and row == 0:
            ax.legend(fontsize=7, ncol=2, loc='lower right', framealpha=0.8)

        ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('LOO-CV: Observed vs Predicted\n'
             'OLS (top row) vs Random Forest (bottom row) — points coloured by station\n'
             'Negative R² = shared weather station explains temporal variation only',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.savefig('fig_obs_pred_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_obs_pred_comparison.png')